# Data Provenance 2 Workflow Provenance Notebook

_by Max Schröder$^1$, Sascha Genehr$^2$, Tanja Auge$^3$, Susanne Staehlke$^4$, Meike Klettke$^3$ and Frank Krüger$^2$_

$^1$ University Library, University of Rostock, Rostock <br>
$^2$ Data Science and Machine Learning Group, Wismar University of Applied Sciences, Wismar <br>
$^3$ Faculty for Informatics and Data Science, University of Regensburg, Regensburg, <br>
$^4$ Department of Cell Biology, Rostock University Medical Center, Rostock

**Abstract**:
This notebook demonstrates how Jupyter notebooks can be used to transfer data provenance output in the form of XML queries into workflow provenance using the DWprov ontology. The corresponding workflow entities are provided as parameters to the transformation function.

### Provenance Graph


In [ ]:
import xml.etree.ElementTree as ET
from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, XSD

# PROV Namespaces
PROV = Namespace("http://www.w3.org/ns/prov#")
BASE = Namespace("http://eln-provenance.elaine.uni-rostock.de/data-provenance/")
BFO  = Namespace("http://purl.obolibrary.org/obo/bfo.owl")
DWPROV   = Namespace("https://w3id.org/DWprov#")

In [ ]:
def shorten(uri):
    uri = str(uri)
    if "prov#" in uri:
        return "prov:" + uri.split("#")[-1]
    if "rdf-schema#" in uri:
        return "rdfs:" + uri.split("#")[-1]
    if "application#" in uri:
        return ":" + uri.split("#")[-1]
    #if "prov-ext#" in uri:
    #    return uri.split("#")[-1]
    if "rdf-syntax-ns#" in uri:
        return "rdf:" + uri.split("#")[-1]
    if "w3id.org/DWprov#" in uri:
        return "dwprov:" + uri.split("#")[-1]
    return uri

In [ ]:
import re

def load_xml(path):
    tree = ET.parse(path)
    return tree.getroot()

def create_graph():
    g = Graph()
    g.bind("xsd", XSD)
    g.bind("prov", PROV)
    g.bind("rdfs", RDFS)
    g.bind("bfo", BFO)
    g.bind("dwprov", DWPROV)
    g.bind("", BASE)
    return g

def add_sources(g, xml_root, only_selected_attrs=False, selected_attrs=None, wf_source_entity_ids=None):
    for atom in xml_root.findall(".//originalinstance/atom"):
        source_id = atom.attrib.get("id")

        if not source_id:
            print("⚠️ atom without an ID")
            continue

        source = URIRef(BASE + source_id)

        comment_parts = []
        source_type = None

        for const in atom.findall("./constant"):
            attr_name = const.attrib['name']
            attr_value = const.attrib['value']

            if attr_name == 'type':
                source_type = attr_value

            if only_selected_attrs and selected_attrs:
                if attr_name not in selected_attrs:
                    continue

            comment_parts.append(f"{attr_name}:{attr_value}")

        comment = f" {atom.attrib['name']} (original): " + ", ".join(comment_parts)

        g.add((source, RDF.type, PROV.Entity))
        g.add((source, RDFS.comment, Literal(comment)))

        for wf_entity in wf_source_entity_ids[source_type]:
            g.add((wf_entity, URIRef('http://schema.org/hasPart'), source))

def add_activities(g, xml_root, wf_activity_id):
    for atom in xml_root.findall(".//provenance/atom"):
        activity_id = atom.attrib['id']
        activity = URIRef(BASE + f"activity_{activity_id}")

        g.add((activity, RDF.type, PROV.Activity))
        g.add((wf_activity_id, URIRef('http://schema.org/hasPart'), activity))

        # -------------------------
        # WHY (witnesses)
        # -------------------------
        witnesses = []

        for witness in atom.findall("./why/witness_base/witness"):
            #tuple_ids = []
            for tuple_elem in witness.findall("./tuple"):
                tuple_id = tuple_elem.attrib.get("id")
                if tuple_id:
                    witnesses.append(tuple_id)

        if witnesses:
            witness_str = "{" + ",".join(f"{{{tid}}}" for tid in witnesses) + "}"
            why_witness = BNode()
            g.add((why_witness, PROV.value, Literal(witness_str)))
            g.add((activity, DWPROV.hasWhyDataProvenance, why_witness))

        # -------------------------
        # WHERE
        # -------------------------
        for tuple_elem in atom.findall("./where/tuples/tuple"):
            tuple_id = tuple_elem.attrib.get("id")
            if tuple_id:
                where_location = BNode()
                g.add((where_location, PROV.value, Literal(tuple_id)))
                g.add((activity, DWPROV.hasWhereDataProvenance, where_location))

                if activity_id.startswith("result_"):
                    used_node = URIRef(BASE + tuple_id)
                    g.add((used_node, RDF.type, PROV.Entity))
                    g.add((activity, PROV.used, used_node))

        # -------------------------
        # HOW
        # -------------------------
        how_exprs = [
            h.attrib['expr']
            for h in atom.findall("./how/polynom")
            if 'expr' in h.attrib
        ]

        if how_exprs:
            how_string = ", ".join(how_exprs)

            how_polynomial = BNode()
            g.add((how_polynomial, PROV.value, Literal(how_string)))
            g.add((activity, DWPROV.hasHowDataProvenance, how_polynomial))

def add_targets(g, xml_root, wf_result_entity_id):
    for atom in xml_root.findall(".//instance/atom"):
        target_id = atom.attrib.get('id')

        if not target_id:
            continue

        target = URIRef(BASE + target_id)

        comment_parts = []
        value_parts = []

        for const in atom.findall("./constant"):
            comment_parts.append(f"{const.attrib['name']}:{const.attrib['value']}")
            if const.attrib['name'] not in ['time', 'type']:
                value_parts.append(const.attrib['value'])

        comment = f"{atom.attrib['name']}: " + ", ".join(comment_parts)
        value = ", ".join(value_parts)

        g.add((target, RDF.type, PROV.Entity))
        g.add((target, RDFS.comment, Literal(comment)))
        g.add((target, PROV.value, Literal(value)))
        g.add((target, PROV.wasGeneratedBy, URIRef(BASE + f"activity_{target_id}")))
        g.add((target, URIRef('http://schema.org/isPartOf'), wf_result_entity_id))

def _extract_region_attrs(query_str: str) -> list[str]:
    region_regex = re.compile(r"SELECT .*? '([^']+)' AS region", re.S | re.I)

    allowed = {"R1_1", "R2_1", "R3_1"}
    regions = set()
    for match in region_regex.finditer(query_str):
        region_literal = match.group(1).strip()
        if region_literal in allowed:
            m = re.match(r"R(\d+)_\d+", region_literal)
            if m:
                num = m.group(1)
                regions.add(f"r1_{num}")

    return sorted(regions, key=lambda x: int(x.split("_")[1]))

def build_rdf_from_xml(xml_path: str, select: bool, query: str | None = None, wf_activity_id: str | None = None, wf_result_entity_id: str | None = None, wf_source_entity_ids: list | None = None):
    xml_root = load_xml(xml_path)

    g = create_graph()

    if not select:
        add_sources(g, xml_root, only_selected_attrs=False, wf_source_entity_ids=wf_source_entity_ids)
    else:
        if query is None:
            add_sources(g, xml_root, only_selected_attrs=False, wf_source_entity_ids=wf_source_entity_ids)
        else:
            region_attrs = _extract_region_attrs(query)

            selected_attrs = ["provid", "time", "type"] + region_attrs

            add_sources(g, xml_root,
                        only_selected_attrs=True,
                        selected_attrs=selected_attrs,
                        wf_source_entity_ids=wf_source_entity_ids)


    add_activities(g, xml_root, wf_activity_id)
    add_targets(g, xml_root, wf_result_entity_id)

    return g

In [ ]:
select = False # print only select attributes
query = None
activity_id = URIRef("https://eln-provenance.elaine.uni-rostock.de/jupyter/CA-Imaging-Stimulation-Analysis.ipynb/execution")
result_entity_id = URIRef("https://eln-provenance.elaine.uni-rostock.de/jupyter/CA-Imaging-Stimulation-Analysis.ipynb/cell/a8f2c0185946f7122b4ab3cee9bf3bf08e7871661ffbac4ade61c6636f417271")
source_entity_ids = {
    "1V_20Hz": [
        URIRef("semantic_eln_export/transformed/ro-crate_942/Data/04_Zeitserie-nach-Stimulation_1V_20Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1021/Data/01_Zeitserie-nach-Stimulation_1V-20-Hz_def-Bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1022/Data/04_Zeitserie-nach-Stimulation_1V-20Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1042/Data/01_Zeitserie-nach-Stimulation_1V-20-Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1124/Data/05_nach-Stimulation_1V-20Hz_def.-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/04_Zeitserie-nach-Stimulation_1V-20-Hz_def-bereiche-1-89.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/04_Zeitserie-nach-Stimulation_1V-20-Hz_def-bereiche-90-240.csv")
    ],
    "1V_7.9Hz": [
        URIRef("semantic_eln_export/transformed/ro-crate_942/Data/03_Zeitserie-nach-Stimulation_1V_7.9Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1021/Data/05_Zeitserie-nach-Stimulation_1V-7.9Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1022/Data/03_Zeitserie-nach-Stimulation_1V-7.9Hz_def.-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1042/Data/02_Zeitserie-nach-Stimulation_1V-7.9-Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/05_Zeitserie-nach-Stimulation_1V-7.9-Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1124/Data/02_Zeitserie-nach-Stimulation_1V-7.9Hz_def.-Bereiche.csv")
    ],
    "5V_20Hz": [
        URIRef("semantic_eln_export/transformed/ro-crate_942/Data/05_Zeitserie-nach-Stimulation_5V_20Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1021/Data/02_Zeitserie-nach-Stimulation_5V-20-Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1042/Data/05_Zeitserie-nach-Stimulation_5V-20-Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1124/Data/04_nach-Stimulation_5V-20Hz_def-bereich.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1022/Data/05_Zeitserie-nach-Stimulation_5V-20Hz_def-bereiche01-98.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1022/Data/05_Zeitserie-nach-Stimulation_5V-20Hz_def-bereiche-100-240.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/03_Zeitserie-nach-Stimulation_5V-20-Hz_def-Bereiche-1-88.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/03_Zeitserie-nach-Stimulation_5V-20-Hz_def-Bereiche-89-240.csv")
    ],
    "5V_7.9Hz": [
        URIRef("semantic_eln_export/transformed/ro-crate_942/Data/02_Zeitserie-nach-Stimulation_5V_7.9Hz_def-Bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1021/Data/04_Zeitserie-nach-Stimulation_5V-7.9Hz_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1022/Data/06_Zeitserie-nach-Stimulation_5V-7.9Hz-Wdh_def-Bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1042/Data/04_Zeitserie-nach-Stimulation_5V-7.9-Hz_def-Bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1124/Data/03_Zeitserie-nach-Stimulation_5V-7.9Hz_def-Berieche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/01_Zeitserie-nach-Stimulation_5V-7.9-Hz_def-bereiche-1-81.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/01_Zeitserie-nach-Stimulation_5V-7.9-Hz_def-bereiche-82-240.csv")
    ],
    "US": [
        URIRef("semantic_eln_export/transformed/ro-crate_942/Data/06_Zeitserie-nach-Stimulation_Kontrolle_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1021/Data/03_Zeitserie-nach-Stimulation_Kontrolle_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1022/Data/01_Zeitserie-nach-Stimulation_Kontrolle_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1042/Data/03_Zeitserie-nach-Stimulation_Kontrolle_def-bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1071/Data/02_Zeitserie-nach-Stimulation_Kontrolle_def-Bereiche.csv"),
        URIRef("semantic_eln_export/transformed/ro-crate_1124/Data/01_Zeitserie-nach-Stimulation_Kontrolle_def-Bereiche.csv")
    ],
}

g = build_rdf_from_xml("provenance_output.xml", select, query, activity_id, result_entity_id, source_entity_ids)

print(f"Number of triples: {len(g)}")

g.serialize("output.ttl", format="ttl")
print("\n✅ RDF saved as output.ttl")

Number of triples: 65

✅ RDF saved as output.ttl


In [24]:
from rdflib import Graph
import subprocess

short = True # shortened namespaces

g = Graph()
g.parse("output.ttl", format="ttl")

dot_file = "graph.dot"
png_file = "graph.png"

with open("graph.dot", "w") as f:
    f.write("digraph G {\n")

    for s, p, o in g:
        s_short = shorten(s) if short else str(s)
        p_short = shorten(p) if short else str(p)
        o_short = shorten(o) if short else str(o)

        f.write(f'    "{s_short}" -> "{o_short}" [label="{p_short}"];\n')

    f.write("}\n")

print("✅ DOT file created")

try:
    subprocess.run(["dot", "-Tpng", dot_file, "-o", png_file], check=True)
    print("✅ PNG file created")
except FileNotFoundError:
    print("❌ Graphviz is not installed (dot is missing)")
except subprocess.CalledProcessError:
    print("❌ Error creating the PNG")

✅ DOT file created


dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.54969 to fit


✅ PNG file created
